### **ACID - PART 4a - Compute background function**

# --- --- ---

### This notebook computes and saves a background function. Part 4b notebook imports the background function and uses it for illumination correction.

### Skeep this notebook if a background function has already been computed and saved.

# --- --- ---

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/03/18


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [ ]:
# Import required modules
from collections.abc import Sequence
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import napari
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
from utils.fov_axis_utils import get_fov_ch_shape
from utils.save_image import tifffile_save_ometiff
from image_processing.calculate_background_function import (
    import_fov,
    calculate_background_function,
    calculate_bg_funct_per_condition,
    get_polyfit_bg_funct_channel,
    compute_simple_background,
)

### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
# metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"
# fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\background"
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\background"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part3 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
# metadata_file_name = "default"
metadata_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# indicate whether or not to save the average background function as an image for comparison
# NOTE: if True, the option is only available when strategy 1 is used for calculating the background function (i.e.
# one background image is calculated per each channel by pooling together all fields of view)
# and the image will be saved in the secondary_output directory
save_avg_background_function_image = False

# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column = "is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val = 1

# # --- parameters to select the non flagged fields of view ---
# indicate the name of the column indicating whether the row belongs is or is not flagged
flag_column = "flag"

# indicate the value signalling that a row (aka a field of view) is flagged
# NOTE: any value which is not this value will be considered as unflagged - only unflagged fields of view
# will be used in the background function computation
flag_value = 1


# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"  # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None  # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = "_"

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format = "%Y%m%d"

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for background function calculation ---
# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# background function computation method - this is the method to use for calculating the
# background function. Possible options are:
# "median" - the background function will be calculated as the median of the pixel values across the fields of view (after applying the flag and train set filters)
# "mean" - the background function will be calculated as the mean of the pixel values across the fields of view (after applying the flag and train set filters)
background_function_avg_method = "median"

# method to use for fitting the background function - this is the method to use for fitting the
# background function after calculating the average background function using the method indicated above
# (median or mean).

# Possible options are:

# "simple" - the background function will be fitted using a rolling ball algorithm followed by a gaussian smoothing
# (ref to image_processing.calculate_background_function.compute_simple_background for more details) - this is the
# default method, and recommended for most use cases, as it is more robust to outliers and can capture
# more complex background functions. If the background function is calculated using the "simple" method,
# the parameters indicated below (ball_radius, white_background, _rb_kwargs, invert_kwargs, gau_smooth,
# gaussian_kwargs, convolve_kwargs) will be used for calculating the background function

# "polyfit" - the background function will be fitted using a polynomial surface
# (ref to image_processing.calculate_background_function.get_polyfit_background for more details) - this method
# is less robust to outliers and may not capture complex background functions, but it is faster to compute.
# If the background function is calculated using the "polyfit" method, the parameters indicated below
# (polynomial_order_x and polynomial_order_y) will be used for calculating the background function.
background_fit_method = "simple"  # pick between "simple" and "polyfit"

# polynomial degree to use for fitting the polynomial surface after calculating the background
# function using the method indicated above (median or mean).
# Recommended to use 1 or 2 as degree - if the degree is too high, the fitted
# surface may overfit the background function and not generalize well to other fields of view
# NOTE: a background function is calculated for each channel.
polynomial_order_x = (1, 1, 1, 1, 1)
polynomial_order_y = (1, 1, 1, 1, 1)

# # If None, all coefficients up to maxiumum kx, ky, ie. up to and including x^kx*y^ky, are considered.
# # If int, coefficients up to a maximum of kx+ky <= order are considered.
# # Ref to image_processing.calculate_background_function.get_polyfit_background_function for more details.
# order = None

# radius of the rolling ball to be used for background function calculation when background_fit_method is "simple"
# NOTE: the same radius will be used for all channels
ball_radius = 200

# indicate whether to use a white background when applying the rolling ball algorithm
# for background function calculation when background_fit_method is "simple"
# NOTE: the same value will be used for all channels. Recommended False.
white_background = (False, False, False, False, True)

# kwargs for the rolling ball algorithm - these are the kwargs to be passed to the
# function implementing the rolling ball algorithm when background_fit_method is "simple" - ref to image_processing.calculate_background_function.compute_simple_background for more details
# NOTE: the same values will be used for all channels
_rb_kwargs = None

# kwargs for inverting the image values before applying the rolling ball algorithm - these are the kwargs to be
# passed to the function implementing the rolling ball algorithm when background_fit_method is "simple"
# ref to image_processing.calculate_background_function.compute_simple_background for more details
# NOTE: the same values will be used for all channels
invert_kwargs = None

# size of the gaussian smoothing to be applied to the background function after applying the rolling ball
# algorithm when background_fit_method is "simple"
# Possible options are:
# - if None, no gaussian smoothing will be applied
# - if int, the size of the gaussian kernel will be (gau_smooth, gau_smooth)
# - if an array is passed, the array will be used as the gaussian kernel. NOTE: as of 2026/03/17 the behaviour of this
# option when saving the metadata hasn't been defined.
# NOTE: the same value will be used for all channels - recommended 20.
# ref to image_processing.calculate_background_function.compute_simple_background for more details
gau_smooth = 100

# kwargs for the gaussian smoothing to be applied to the background function after applying the rolling ball
# algorithm when background_fit_method is "simple" - these are the kwargs to be passed to
# NOTE: the same value will be used for all channels - recommended 20.
gaussian_kwargs = None

# kwargs for the convolution to be applied to the background function after applying the rolling ball
# algorithm when background_fit_method is "simple"
# NOTE: the same value will be used for all channels - recommended 20.
convolve_kwargs = None

# kwargs for the dtype, number of jobs, and map function when calculating the background function
# NOTE: the same value will be used for all channels - recommended 20.
dtype = None

# number of jobs to use for parallel processing when calculating the background function
# if None, the number of jobs will be set to the number of CPU cores
n_workers = 5

# kwargs for the map function to be used for parallel processing when calculating the background function
map_kwargs = None

# indicate whether to print verbose messages during the background function calculation using the method "simple"
verbose = True


# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# well column name in the metadata dataframe
well_column_name = "well"

# plate column name in the metadata dataframe
plate_column_name = "experiment"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
gridpos_column_name = "scene_name"

# null value to be used when a result can't be computed
null_value = np.nan

# kwargs of np.zeros - np.zeros is used to generate a container array storing all images used to
# calculate the background function - by passing kwargs to it one can define the format of the
# background function (e.g. the dtype) - ref to https://numpy.org/doc/stable/reference/generated/numpy.zeros.html
np_zero_kwargs = None

# sample dataframe - this is used for testing purposes
# if True, only a fraction of the fields of view will be used for calculating the background function, and the results will be saved in a separate directory. This is useful for testing the pipeline on a smaller dataset before running it on the full dataset. The fraction of fields of view to use can be defined using the parameter sample_fraction, and additional kwargs for sampling the dataframe can be passed using sample_kwargs (ref to pandas.DataFrame.sample documentation).
sample_df = False
sample_fraction = 0.5
sample_kwargs = {"random_state": 42}

# axis along which fields of view are stacked before calculating the background function
axis_calc_bg = -1

# indicate whether to print verbose messages during the background function calculation
# the same parameter is passed to all the functions used for calculating the background function,
# so that the messages printed during the different steps of the background function calculation are consistent.
verbose_calc_bg = True


# --- parameters for saving metadata within the background function image ---
# background function image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the background function image.
# The entry indicates the date when the background function was calculated
background_img_meta_date_name = "background_funct_date_yymmdd"

# background function image - date format in metadata - this is the format to use for indicating
# the date when the background function was calculated in the metadata saved within the background function image
background_img_meta_date_format = "%y%m%d"

# background function image - project name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# name of the project (e.g. ACID)
background_img_meta_project_name = "project_name"

# background function image - method name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# method used for calculating the background function (e.g. median or mean)
background_img_meta_avg_method_name = "background_funct_avg_method"

# background function image - polynomial degree name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# degree of the polynomial used for fitting the background function (if polynomial fitting is applied)
background_img_meta_poly_degree_name = "background_funct_poly_degree"

# background function image - ball radius name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# radius of the rolling ball used for calculating the background function when background_fit_method is "simple"
background_img_meta_ballradius_name = "background_funct_ball_radius"

# background function image - white background name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates whether a white
# background was used when applying the rolling ball algorithm for calculating the background function
# when background_fit_method is "simple"
background_img_meta_whitebg_name = "background_funct_white_bg"

# background function image - gaussian smoothing name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the size of the
# gaussian smoothing applied to the background function after applying the rolling ball algorithm
# for calculating the background function when the background_fit_method is "simple"
background_img_meta_gausmooth_name = "background_funct_gau_smooth_size"


# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True


# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# channel name separator - this is the separator to use for separating the channel name from the rest of
# the column name to be added to the metadata dataframe.
ch_name_separator = "-"

# background function metadata dataframe - method column name - this is the name of the column to be
# added to the metadata dataframe to indicate the method used for calculating the background function
background_df_avg_method_clm_name = f"background{column_name_separator}funct{column_name_separator}avg{column_name_separator}method"

# background function metadata dataframe - polynomial degree column name - this is the name of the column to be added to
# the metadata dataframe to indicate the degree of the polynomial used for fitting the background function
background_df_poly_order_x_clm_name = f"background{column_name_separator}funct{column_name_separator}order{column_name_separator}x"
background_df_poly_order_y_clm_name = f"background{column_name_separator}funct{column_name_separator}order{column_name_separator}y"

# background function metadata dataframe - computation date column name - this is the name of the column to be added
# to the metadata dataframe to indicate the day when the background function was calculated
background_df_date_clm_name = (
    f"background{column_name_separator}funct{column_name_separator}date"
)

# background function metadata dataframe - computation date format - this is the format to be used for indicating the
# date when the background function was calculated. This is used for saving the date in the metadata dataframe
background_df_meta_date_format = "%y%m%d"

# background function metadata dataframe - ball radius column name - this is the name of the column to be added to
# the metadata dataframe to indicate the radius of the rolling ball used for calculating the background function when
# background_fit_method is "simple"
background_df_meta_ballradius_name = f"background{column_name_separator}funct{column_name_separator}ball{column_name_separator}radius"

# background function metadata dataframe - white background column name - this is the name of the column to be added
# to the metadata dataframe to indicate whether a white background was used when applying the rolling ball algorithm
# for calculating the background function when background_fit_method is "simple"
background_df_meta_whitebg_name = f"background{column_name_separator}funct{column_name_separator}white{column_name_separator}bg"

# background function metadata dataframe - gaussian smoothing column name - this is the name of the column to
# be added to the metadata dataframe to indicate the size of the gaussian smoothing applied to the background
# function after applying the rolling ball algorithm for calculating the background function when
# background_fit_method is "simple"
background_df_meta_gausmooth_name = f"background{column_name_separator}funct{column_name_separator}gau{column_name_separator}smooth"


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = (
    False  # if False, the index will not be saved as a separate column in the csv file
)

# background function image name - date format - this is the format to use for
# indicating the date when the background function was calculated in the background function image name
background_img_name_date_format = "%Y%m%d"

# background function image name - savingword - this is the word to use in the background
# function image name to indicate that the file is a background function image
background_img_savingword = "background"
non_polyfit_background_img_savingword = f"bg{column_name_separator}nofit"

# background function image name - file suffix - this is the suffix to use for the
# background function image file name
background_img_file_suffix = ".ome.tif"

# background function image - data type - this is the data type to use for saving the background function image
background_img_dtype = np.float32

# background function image - photometric - this is the photometric to use for saving the background function image
background_img_photometric = "minisblack"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4a.csv"

# processing metadata dataframe name - date format
metadata_date_format = "%Y%m%d"

# hyperparameters dataframe name - date format
hyperparameters_date_format = "%Y%m%d-%H%M%S"

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}4a.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = (
    True  # if the secondary output directory already exists, do not raise an error
)

### Create output directory and secondary output directory if they don't exist

##### Output directory stores the computed background function
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [ ]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)

#### Open the metadata dataframe - this is expected to be the output of part 3

Run the following cell.

Don't modify the following cell.

In [ ]:
# check if using the default metadata data frame (the most recently saved)
if (
    metadata_file_name == None
    or metadata_file_name.lower() == "default"
    or metadata_file_name == ""
):
    # import target files in the metadata_directory
    metadata_files = listdirNHF(
        metadata_directory,
        target=default_metadata_file_target,
        exclude=default_metadata_file_exclude,
    )

    # get the default metadata file
    metadata_file_name = default_file_name(
        file_list=metadata_files,
        from_file_name=metadata_from_file_name,
        directory_path=metadata_directory,
        separator=metadata_default_separator,
        date_position=metadata_default_date_position,
        date_format=metadata_default_date_format,
        reverse=metadata_default_reverse,
    )

    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df

In [ ]:
metadata_df.head()

#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [ ]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), (
    "Not all rows in metadata_df belong to the train set."
)

# Display the metadata dataframe
metadata_df

#### Select non flagged fields of view

Run the following cell.

Don't modify the following cell.

In [ ]:
# Select only non-flagged rows and update metadata_df
metadata_df_ok = metadata_df[metadata_df[flag_column] != flag_value]

# assert proper selection of train set
assert metadata_df_ok.shape[0] > 0, (
    "No row remains in metadata_df after removing the flagged ones."
)
assert all(metadata_df_ok[flag_column] != flag_value), (
    "Some row in metadata_df_ok are flagged after selection."
)

# Display the metadata dataframe
metadata_df_ok

# metadata_df_ok = metadata_df.copy()
# metadata_df_ok

#### Get the shape and the number of channels of the fields of view - NOTE: it is assumed that all fields of view in the dataset have the same shape and number of channels

Run the following cell.

Don't modify the following cell.

In [ ]:
# get the shape of the individual fields of view, the number of channels and the shape of individual channels
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(
    metadata_df_ok,
    fov_directory,
    fov_clm=fov_column_name,
    channel_axis=channel_axis,
    null_value=null_value,
)

#### Strategy 1 - Calculate a background function by averaging all the fields of view in the dataset

The following cell:
1) import all the fields of view in the dataset
2) stack all the fields of view in a single array.
3) average all the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_avg_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:
# import all fields of view in the dataset and stack them into a single array
container_arr = import_fov(
    df=metadata_df_ok,
    fov_dir=fov_directory,
    fov_clm=fov_column_name,
    fov_shape=fov_shape,
    np_zero_kwargs=np_zero_kwargs,
    verbose=verbose_calc_bg,
    sample_df=sample_df,
    sample_fraction=sample_fraction,
    sample_kwargs=sample_kwargs,
)

# initialize a list for storing the background function calculated for each channel
background_function_collection = []

# unstack the container array along the channel axis to get a list of arrays
# each containing the pixel values of the fields of view for a single channel
# this is used for calculating the background function per channel
container_arr_ch = np.unstack(container_arr, axis=channel_axis)

# iterate over the list of arrays and calculate the background function for each channel
for ch_idx, container_arr_ch_i in enumerate(container_arr_ch):
    print(container_arr_ch_i.shape)
    # calculate the background function for individual channels by mean/median average projection
    ch_i_background_function = calculate_background_function(
        container_arr=container_arr_ch_i,
        method=background_function_avg_method,
        axis=axis_calc_bg,
        verbose=verbose_calc_bg,
    )
    print(ch_i_background_function.shape)

    # add the calculated background function for the current channel to the list for storing the background function
    background_function_collection.append(ch_i_background_function)

# stack the background function calculated for each channel into a single array
background_function = np.stack(background_function_collection, axis=channel_axis)
print(background_function.shape)

#### Visualize the background functions per each channel

Run the following cell.

Don't modify the following cell.

In [ ]:
# istanziate a napari viewer and add the background function to the viewer
napari_viewer = napari.Viewer()

for ch in range(background_function.shape[channel_axis]):
    napari_viewer.add_image(
        background_function.take(ch, axis=channel_axis),
        name=f"background_funct_ch_{ch}",
    )

#### Strategy 2 - Calculate a background function per each well of the dataset

The following cell, iteratively per each well:
    
1) import all the fields of view in the dataset belonging to the well
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_avg_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:
# get the background functions of all the wells in the dataframe and map them into a dictionary
background_functions_per_well = calculate_bg_funct_per_condition(
    df=metadata_df_ok,
    condition_clm=well_column_name,
    fov_dir=fov_directory,
    fov_clm=fov_column_name,
    fov_shape=fov_shape,
    method=background_function_avg_method,
    stack_axis=axis_calc_bg,
    verbose=verbose_calc_bg,
)

#### Visualize the background functions per each well and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the channel to visualize
ch_to_plot = 4

# --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# istanziate a napari viewer and add the background function to the viewer
napari_viewer_1 = napari.Viewer()

for wel_l in background_functions_per_well:
    napari_viewer_1.add_image(
        background_functions_per_well[wel_l].take(ch_to_plot, axis=channel_axis),
        name=f"background_funct_{wel_l}_{ch_to_plot}",
    )

#### Strategy 3 - Calculate a background function per each field of view grid position

Per each well 49 positions are aquired from a 7x7 grid. The following cells analyse the result of computing a background function per each of the 49 position, by averaging across experiemnts, plates and wells.

Precisely, the following cell, iteratively per each grid position:
    
1) import all the fields of view in the dataset belonging to the grid position
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_avg_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [ ]:
# get the background functions of all the wells in the dataframe and map them into a dictionary
background_functions_per_gridpos = calculate_bg_funct_per_condition(
    df=metadata_df_ok,
    condition_clm=gridpos_column_name,
    fov_dir=fov_directory,
    fov_clm=fov_column_name,
    fov_shape=fov_shape,
    method=background_function_avg_method,
    stack_axis=axis_calc_bg,
    verbose=verbose_calc_bg,
)

#### Visualize the background functions per each grid position and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the channel to visualize
ch_to_plot_1 = 4

# --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# istanziate a napari viewer and add the background function to the viewer
napari_viewer_2 = napari.Viewer()

for grid_pos in background_functions_per_gridpos:
    napari_viewer_2.add_image(
        background_functions_per_gridpos[grid_pos].take(
            ch_to_plot_1, axis=channel_axis
        ),
        name=f"background_funct_{grid_pos}_{ch_to_plot_1}",
    )

#### Define the strategy to use for background function calculation

Run the following cell.

MODIFY the following cell.

In [ ]:
# indicate which strategy to use for creating the background function by fitting a
# polynomial surface to the averaged background function
background_function_strategy = 1  # pick between 1, 2 or 3

#### Finalize the background function computation. Two methods are possible (use the parameter background_fit_method to decide):
#### Option 1 (not recommended): fit polynomial surface to background function.
#### Option 2 (recommended and default): use the rolling ball algorithm to calculate the background, then smooth the calculated background using a gaussian kernel.

#### Update and save the metadata_df. NOTE: the update is done on the metadata_df after train data filter and before filtering of the flagged rows.

Run the following cell.

Don't modify the following cell.

In [ ]:
# create the background function by fitting a polynomial surface to the averaged background function and
# according to the strategy indicated above, save the background function as an image
if background_function_strategy == 1:
    if background_fit_method == "polyfit":
        # create the background function by fitting a polynomial surface to the averaged background function
        # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to
        # the maximum kx and ky indicated will be considered
        fit_background_function = get_polyfit_bg_funct_channel(
            background_function=background_function,
            channel_axis=channel_axis,
            kx=polynomial_order_x,
            ky=polynomial_order_y,
            verbose=verbose_calc_bg,
        )
        # create the background function image metadata dictionary
        background_img_metadata_dict = {
            background_img_meta_date_name: datetime.datetime.now().strftime(
                background_img_meta_date_format
            ),
            background_img_meta_project_name: project_name,
            background_img_meta_avg_method_name: background_function_avg_method,
            background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}",
        }  # NOTE: order is hardcoded as None

    elif background_fit_method == "simple":
        # create the background function by using a rolling ball algorithm and gaussian smoothing
        # to the averaged background function
        fit_background_function = compute_simple_background(
            background_function,
            ball_radius=ball_radius,
            white_background=white_background,
            _rb_kwargs=_rb_kwargs,
            invert_kwargs=invert_kwargs,
            gau_smooth=gau_smooth,
            gaussian_kwargs=gaussian_kwargs,
            convolve_kwargs=convolve_kwargs,
            dtype=dtype,
            axis=channel_axis,
            n_workers=n_workers,
            map_kwargs=map_kwargs,
        )

        # create the background function image metadata dictionary
        background_img_metadata_dict = {
            background_img_meta_date_name: datetime.datetime.now().strftime(
                background_img_meta_date_format
            ),
            background_img_meta_project_name: project_name,
            background_img_meta_avg_method_name: background_function_avg_method,
            background_img_meta_ballradius_name: ball_radius,
            background_img_meta_whitebg_name: white_background,
            background_img_meta_gausmooth_name: gau_smooth,
        }

    else:
        raise ValueError("background_fit_method must be either 'polyfit' or 'simple'")

    # create the background function image name
    background_img_name = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{background_img_file_suffix}"

    # save the background function as an image
    tifffile_save_ometiff(
        os.path.join(output_directory, background_img_name),
        data=fit_background_function.astype(background_img_dtype),
        imagej=save_imagej_compatible,
        photometric=background_img_photometric,
        metadata=background_img_metadata_dict,
    )

    # also save the non-polyfit background function as an image for comparison, if required
    if save_avg_background_function_image:
        background_img_name_no_fit = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{non_polyfit_background_img_savingword}{background_img_file_suffix}"
        background_img_metadata_dict_no_fit = {
            background_img_meta_date_name: datetime.datetime.now().strftime(
                background_img_meta_date_format
            ),
            background_img_meta_project_name: project_name,
            background_img_meta_avg_method_name: background_function_avg_method,
        }  # NOTE: order is hardcoded as None

        # save the non-polyfit background function as an image for comparison
        tifffile_save_ometiff(
            os.path.join(secondary_output_directory, background_img_name_no_fit),
            data=background_function.astype(background_img_dtype),
            imagej=save_imagej_compatible,
            photometric=background_img_photometric,
            metadata=background_img_metadata_dict_no_fit,
        )

elif background_function_strategy == 2:
    # iterate over the wells
    for wel_l in background_functions_per_well:
        if background_fit_method == "polyfit":
            # create the background function by fitting a polynomial surface to the averaged background function of the well
            # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to the maximum
            # kx and ky indicated will be considered
            fit_background_function_well = get_polyfit_bg_funct_channel(
                background_function=background_functions_per_well[wel_l],
                channel_axis=channel_axis,
                kx=polynomial_order_x,
                ky=polynomial_order_y,
                verbose=verbose_calc_bg,
            )

            # create the background function image metadata dictionary for the well
            background_img_metadata_dict_well = {
                background_img_meta_date_name: datetime.datetime.now().strftime(
                    background_img_meta_date_format
                ),
                background_img_meta_project_name: project_name,
                background_img_meta_avg_method_name: background_function_avg_method,
                background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}",  # NOTE: order is hardcoded as None
                "well": wel_l,
            }

        elif background_fit_method == "simple":
            # create the background function by using a rolling ball algorithm and gaussian smoothing
            # to the averaged background function of the well
            fit_background_function = compute_simple_background(
                background_functions_per_well[wel_l],
                ball_radius=ball_radius,
                white_background=white_background,
                _rb_kwargs=_rb_kwargs,
                invert_kwargs=invert_kwargs,
                gau_smooth=gau_smooth,
                gaussian_kwargs=gaussian_kwargs,
                convolve_kwargs=convolve_kwargs,
                dtype=dtype,
                axis=channel_axis,
                n_workers=n_workers,
                map_kwargs=map_kwargs,
            )

            # create the background function image metadata dictionary for the well
            background_img_metadata_dict_well = {
                background_img_meta_date_name: datetime.datetime.now().strftime(
                    background_img_meta_date_format
                ),
                background_img_meta_project_name: project_name,
                background_img_meta_avg_method_name: background_function_avg_method,
                background_img_meta_ballradius_name: ball_radius,
                background_img_meta_whitebg_name: white_background,
                background_img_meta_gausmooth_name: gau_smooth,
                "well": wel_l,
            }

        else:
            raise ValueError(
                "background_fit_method must be either 'polyfit' or 'simple'"
            )

        # create the background function image name for the well
        background_img_name_well = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{wel_l}{background_img_file_suffix}"

        # save the background function as an image for the well
        tifffile_save_ometiff(
            os.path.join(output_directory, background_img_name_well),
            data=fit_background_function_well.astype(background_img_dtype),
            imagej=save_imagej_compatible,
            photometric=background_img_photometric,
            metadata=background_img_metadata_dict_well,
        )

elif background_function_strategy == 3:
    # iterate over the grid positions
    for grid_pos in background_functions_per_gridpos:
        if background_fit_method == "polyfit":
            # create the background function by fitting a polynomial surface to the averaged background function of the grid position
            # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up
            # to the maximum kx and ky indicated will be considered
            fit_background_function_gridpos = get_polyfit_bg_funct_channel(
                background_function=background_functions_per_gridpos[grid_pos],
                channel_axis=channel_axis,
                kx=polynomial_order_x,
                ky=polynomial_order_y,
                verbose=verbose_calc_bg,
            )

            # create the background function image metadata dictionary for the grid position
            background_img_metadata_dict_gridpos = {
                background_img_meta_date_name: datetime.datetime.now().strftime(
                    background_img_meta_date_format
                ),
                background_img_meta_project_name: project_name,
                background_img_meta_avg_method_name: background_function_avg_method,
                background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}",  # NOTE: order is hardcoded as None
                "grid_position": grid_pos,
            }

        elif background_fit_method == "simple":
            # create the background function by using a rolling ball algorithm and gaussian smoothing
            # to the averaged background function of the grid position
            fit_background_function = compute_simple_background(
                background_functions_per_well[grid_pos],
                ball_radius=ball_radius,
                white_background=white_background,
                _rb_kwargs=_rb_kwargs,
                invert_kwargs=invert_kwargs,
                gau_smooth=gau_smooth,
                gaussian_kwargs=gaussian_kwargs,
                convolve_kwargs=convolve_kwargs,
                dtype=dtype,
                axis=channel_axis,
                n_workers=n_workers,
                map_kwargs=map_kwargs,
            )

            # create the background function image metadata dictionary for the grid position
            background_img_metadata_dict_gridpos = {
                background_img_meta_date_name: datetime.datetime.now().strftime(
                    background_img_meta_date_format
                ),
                background_img_meta_project_name: project_name,
                background_img_meta_avg_method_name: background_function_avg_method,
                background_img_meta_ballradius_name: ball_radius,
                background_img_meta_whitebg_name: white_background,
                background_img_meta_gausmooth_name: gau_smooth,
                "grid_position": grid_pos,
            }

        else:
            raise ValueError(
                "background_fit_method must be either 'polyfit' or 'simple'"
            )

        # create the background function image name for the grid position
        background_img_name_gridpos = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{grid_pos}{background_img_file_suffix}"

        # save the background function as an image for the grid position
        tifffile_save_ometiff(
            os.path.join(output_directory, background_img_name_gridpos),
            data=fit_background_function_gridpos.astype(background_img_dtype),
            imagej=save_imagej_compatible,
            photometric=background_img_photometric,
            metadata=background_img_metadata_dict_gridpos,
        )

else:
    raise ValueError("background_function_strategy must be 1, 2 or 3")


# # --- --- --- UPDATE THE METADATA DATAFRAME WITH THE BACKGROUND FUNCTION INFORMATION --- --- ---
# add the background function method, polynomial degree and calculation date information to the metadata dataframe
# as new columns
# NOTE: the metadata_df used is the one after selecting the train set but before selecting only the non-flagged
# fields of view
metadata_df[background_df_date_clm_name] = datetime.datetime.now().strftime(
    background_df_meta_date_format
)
metadata_df[background_df_avg_method_clm_name] = background_function_avg_method

if background_fit_method == "polyfit":
    for ch_pos, ch_order in enumerate(zip(polynomial_order_x, polynomial_order_y)):
        metadata_df[
            f"{background_df_poly_order_x_clm_name}{ch_name_separator}{ch_pos}"
        ] = ch_order[0]
        metadata_df[
            f"{background_df_poly_order_y_clm_name}{ch_name_separator}{ch_pos}"
        ] = ch_order[1]

elif background_fit_method == "simple":
    if isinstance(ball_radius, Sequence) and not isinstance(ball_radius, (str, bytes)):
        for ch_pos, ch_ball_radius in enumerate(ball_radius):
            metadata_df[
                f"{background_df_meta_ballradius_name}{ch_name_separator}{ch_pos}"
            ] = ch_ball_radius
    else:
        metadata_df[background_df_meta_ballradius_name] = ball_radius

    if isinstance(white_background, Sequence) and not isinstance(
        white_background, (str, bytes)
    ):
        for ch_pos, ch_white_background in enumerate(white_background):
            metadata_df[
                f"{background_df_meta_whitebg_name}{ch_name_separator}{ch_pos}"
            ] = ch_white_background
    else:
        metadata_df[background_df_meta_whitebg_name] = white_background

    if isinstance(gau_smooth, Sequence) and not isinstance(gau_smooth, (str, bytes)):
        for ch_pos, ch_gau_smooth in enumerate(gau_smooth):
            metadata_df[
                f"{background_df_meta_gausmooth_name}{ch_name_separator}{ch_pos}"
            ] = ch_gau_smooth
    else:
        metadata_df[background_df_meta_gausmooth_name] = gau_smooth

else:
    raise ValueError("background_fit_method must be either 'polyfit' or 'simple'")

# save the updated metadata dataframe as a csv file
metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(
    os.path.join(metadata_directory, metadata_df_name), index=save_csv_index
)

### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {
    "metadata_directory": metadata_directory,
    "fov_directory": fov_directory,
    "output_directory": output_directory,
    "metadata_file_name": metadata_file_name,
    "is_train_column": is_train_column,
    "train_val": train_val,
    "flag_column": flag_column,
    "flag_value": flag_value,
    "default_metadata_file_target": default_metadata_file_target,
    "default_metadata_file_exclude": default_metadata_file_exclude,
    "metadata_from_file_name": metadata_from_file_name,
    "metadata_default_separator": metadata_default_separator,
    "metadata_default_date_position": metadata_default_date_position,
    "metadata_default_date_format": metadata_default_date_format,
    "metadata_default_reverse": metadata_default_reverse,
    "background_function_avg_method": background_function_avg_method,
    "polynomial_order_x": polynomial_order_x,
    "polynomial_order_y": polynomial_order_y,
    "channel_axis": channel_axis,
    "fov_column_name": fov_column_name,
    "well_column_name": well_column_name,
    "plate_column_name": plate_column_name,
    "gridpos_column_name": gridpos_column_name,
    "null_value": null_value,
    "np_zero_kwargs": np_zero_kwargs,
    "sample_df": sample_df,
    "sample_fraction": sample_fraction,
    "sample_kwargs": sample_kwargs,
    "axis_calc_bg": axis_calc_bg,
    "verbose_calc_bg": verbose_calc_bg,
    "background_img_meta_date_name": background_img_meta_date_name,
    "background_img_meta_date_format": background_img_meta_date_format,
    "background_img_meta_project_name": background_img_meta_project_name,
    "background_img_meta_avg_method_name": background_img_meta_avg_method_name,
    "background_img_meta_poly_degree_name": background_img_meta_poly_degree_name,
    "save_imagej_compatible": save_imagej_compatible,
    "column_name_separator": column_name_separator,
    "ch_name_separator": ch_name_separator,
    "background_df_avg_method_clm_name": background_df_avg_method_clm_name,
    "background_df_poly_order_x_clm_name": background_df_poly_order_x_clm_name,
    "background_df_poly_order_y_clm_name": background_df_poly_order_y_clm_name,
    "background_df_date_clm_name": background_df_date_clm_name,
    "background_df_meta_date_format": background_df_meta_date_format,
    "save_file_name_separator": save_file_name_separator,
    "project_name": project_name,
    "save_csv_index": save_csv_index,
    "background_img_name_date_format": background_img_name_date_format,
    "background_img_savingword": background_img_savingword,
    "background_img_file_suffix": background_img_file_suffix,
    "background_img_dtype": background_img_dtype,
    "background_img_photometric": background_img_photometric,
    "metadata_savingword": metadata_savingword,
    "metadata_file_suffix": metadata_file_suffix,
    "metadata_date_format": metadata_date_format,
    "hyperparameters_date_format": hyperparameters_date_format,
    "hyperparameters_savingword": hyperparameters_savingword,
    "hyperparameters_file_suffix": hyperparameters_file_suffix,
    "secondary_output_directory": secondary_output_directory,
    "exist_ok": exist_ok,
    "background_fit_method": background_fit_method,
    "ball_radius": ball_radius,
    "white_background": white_background,
    "_rb_kwargs": _rb_kwargs,
    "invert_kwargs": invert_kwargs,
    "gau_smooth": gau_smooth,
    "gaussian_kwargs": gaussian_kwargs,
    "convolve_kwargs": convolve_kwargs,
    "dtype": dtype,
    "n_workers": n_workers,
    "map_kwargs": map_kwargs,
    "verbose": verbose,
    "background_img_meta_ballradius_name": background_img_meta_ballradius_name,
    "background_img_meta_whitebg_name": background_img_meta_whitebg_name,
    "background_img_meta_gausmooth_name": background_img_meta_gausmooth_name,
    "background_df_meta_ballradius_name": background_df_meta_ballradius_name,
    "background_df_meta_whitebg_name": background_df_meta_whitebg_name,
    "background_df_meta_gausmooth_name": background_df_meta_gausmooth_name,
    "save_avg_background_function_image": save_avg_background_function_image,
}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(
    os.path.join(secondary_output_directory, hyperparameter_saving_name),
    index=save_csv_index,
)